#### EDA 03: Order items table
**File:** `olist_order_items_dataset.csv`  |  **Goal:** understand the item lines of each order, using this table only. 

#### Load and first look
**Question:** how big is the table and what does one row stand for?

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)

RAW = Path("../../data/raw")                      # path from the notebook folder to the raw data
items_raw = pd.read_csv(RAW / "olist_order_items_dataset.csv")
items = items_raw.copy()                          # all changes happen on the copy, raw stays untouched

print("shape:", items.shape)
items.head(3)

shape: (112650, 7)


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.9,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.9,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.0,17.87


- 112,650 rows and 7 columns: order, line number, product, seller, shipping deadline, price and freight.
- `order_item_id` is the line number inside an order (1, 2, 3...). It is not an item identifier.
- One row is one item line, not one order. The orders table has 99,441 rows (raw data summary), so this table has more rows than orders.

#### Data types
**Question:** are the columns in a usable type?

In [2]:
print(items.dtypes)

# The shipping deadline is a date stored as text
items["shipping_limit_date"] = pd.to_datetime(items["shipping_limit_date"])
items["shipping_limit_date"].agg(["min", "max"])

order_id                   str
order_item_id            int64
product_id                 str
seller_id                  str
shipping_limit_date        str
price                  float64
freight_value          float64
dtype: object


min   2016-09-19 00:15:34
max   2020-04-09 22:35:08
Name: shipping_limit_date, dtype: datetime64[us]

- All columns load with a sensible type. Only `shipping_limit_date` is text and is converted to a datetime.
- The deadline runs from 2016-09-19 to 2020-04-09. The orders data ends in 2018, so a date in 2020 is suspicious (see the shipping deadline section).

#### Keys and duplicates
**Question:** what identifies a row, and is any row repeated?

In [3]:
print("fully duplicated rows             :", items.duplicated().sum())
print("repeated (order_id, order_item_id):", items.duplicated(["order_id", "order_item_id"]).sum())

fully duplicated rows             : 0
repeated (order_id, order_item_id): 0


- No duplicated rows. The pair (`order_id`, `order_item_id`) is unique, so it is the key of this table.

#### Missing values
**Question:** which columns have gaps?

In [4]:
# Column-wise missing count and percentage, for every column (zeros included)
missing = pd.DataFrame({"n_rows": len(items),
                        "n_missing": items.isna().sum(),
                        "pct_missing": (items.isna().mean() * 100).round(2)})
display(missing.sort_values("n_missing", ascending=False))

,n_rows,n_missing,pct_missing
order_id,112650,0,0.0
order_item_id,112650,0,0.0
product_id,112650,0,0.0
seller_id,112650,0,0.0
shipping_limit_date,112650,0,0.0
price,112650,0,0.0
freight_value,112650,0,0.0


- No column has a missing value. Nothing to impute or flag in this table.

#### Counts inside this table
**Question:** how many distinct orders, products and sellers does the table hold?

In [5]:
print("item lines        :", len(items))
print("distinct orders   :", items["order_id"].nunique())
print("distinct products :", items["product_id"].nunique())
print("distinct sellers  :", items["seller_id"].nunique())

item lines        : 112650
distinct orders   : 98666
distinct products : 32951
distinct sellers  : 3095


- 112,650 lines belong to 98,666 distinct orders, 32,951 products and 3,095 sellers.
- The orders table has 99,441 rows (raw data summary), so up to 775 orders may have no item lines at all.
- Whether those orders exist, and whether every product and seller id exists in its own table, is checked after the joins.

#### Lines per order
**Question:** how many lines does an order have?

In [6]:
lines_per_order = items.groupby("order_id").size()

# Group 6 and more together for display
bucket = lines_per_order.clip(upper=6)
dist = bucket.value_counts().sort_index().rename("n_orders").to_frame()
dist.index = [str(i) if i < 6 else "6+" for i in dist.index]
dist.index.name = "lines_per_order"
dist["pct_orders"] = (dist["n_orders"] / len(lines_per_order) * 100).round(2)
display(dist)

multi = lines_per_order[lines_per_order > 1]
print("orders with more than 1 line :", len(multi), f"({len(multi) / len(lines_per_order) * 100:.1f}% of orders in this table)")
print("lines inside those orders    :", multi.sum(), f"({multi.sum() / len(items) * 100:.1f}% of all lines)")
print("maximum lines in one order   :", lines_per_order.max())

,n_orders,pct_orders
lines_per_order,,
1,88863,90.06
2,7516,7.62
3,1322,1.34
4,505,0.51
5,204,0.21
6+,256,0.26


orders with more than 1 line : 9803 (9.9% of orders in this table)
lines inside those orders    : 23787 (21.1% of all lines)
maximum lines in one order   : 21


- 88,863 orders (90.1%) have a single line.
- 9,803 orders (9.9%) have more than one line, but they hold 23,787 lines, which is 21.1% of all lines.
- The largest order has 21 lines.
- Any statistic computed on lines (average price, share of freight) gives bigger orders more weight than an order-level statistic does.

#### Sellers per order
**Question:** can an order have more than one seller? This decides how to answer "which seller is behind this order".

In [8]:
sellers_per_order = items.groupby("order_id")["seller_id"].nunique()
seller_dist = sellers_per_order.value_counts().sort_index().rename("n_orders").to_frame()
seller_dist.index.name = "sellers_per_order"
seller_dist["pct_orders"] = (seller_dist["n_orders"] / len(sellers_per_order) * 100).round(2)
display(seller_dist)

# Seller value share inside each order (price only)
seller_value = items.groupby(["order_id", "seller_id"], as_index=False)["price"].sum()
seller_value["share"] = seller_value["price"] / seller_value.groupby("order_id")["price"].transform("sum")

multi_ids = sellers_per_order[sellers_per_order > 1].index
top_share = seller_value[seller_value["order_id"].isin(multi_ids)].groupby("order_id")["share"].max()
print("multi-seller orders                                          :", len(multi_ids))
print("... where the biggest seller holds 50% or more of the value :", f"{(top_share >= 0.5).mean() * 100:.1f}%")

,n_orders,pct_orders
sellers_per_order,,
1,97388,98.70
2,1219,1.24
3,54,0.05
4,3,0.00
5,2,0.00


multi-seller orders                                          : 1278
... where the biggest seller holds 50% or more of the value : 97.5%


- 97,388 orders (98.7%) have a single seller, so the seller is clear for almost all orders.
- 1,278 orders (1.3%) have 2 to 5 sellers.
- In 97.5% of those, one seller holds at least half of the order value.

#### Price and freight
**Question:** what do price and freight look like, and are there impossible values?

In [9]:
items["freight_ratio"] = items["freight_value"] / items["price"]       # freight as a share of the item price

items[["price", "freight_value", "freight_ratio"]].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]).T.round(2)

,count,mean,std,min,1%,5%,50%,95%,99%,max
price,112650.0,120.65,183.63,0.85,9.99,17.00,74.99,349.90,890.00,6735.00
freight_value,112650.0,19.99,15.81,0.00,4.42,7.78,16.26,45.12,84.52,409.68
freight_ratio,112650.0,0.32,0.35,0.00,0.02,0.05,0.23,0.87,1.55,26.24


In [10]:
print("lines with price <= 0       :", (items["price"] <= 0).sum())
print("lines with freight == 0     :", (items["freight_value"] == 0).sum(), f"({(items['freight_value'] == 0).mean() * 100:.2f}%)")
print("lines with freight > price  :", (items["freight_value"] > items["price"]).sum(), f"({(items['freight_value'] > items['price']).mean() * 100:.1f}%)")

lines with price <= 0       : 0
lines with freight == 0     : 383 (0.34%)
lines with freight > price  : 4124 (3.7%)


- Price: median 74.99, mean 120.65, 99th percentile 890, maximum 6,735. The mean is far above the median, so the distribution is strongly right-skewed.
- Freight: median 16.26, mean 19.99, maximum 409.68.
- No price is zero or negative. 383 lines (0.34%) have free freight. 4,124 lines (3.7%) have freight higher than the price.
- Free freight is a real case (a promotion or free shipping), not a missing value.

#### Shipping deadline
**Question:** is the seller's shipping deadline in a believable range, and does it differ between lines of one order?

In [12]:
deadline = items["shipping_limit_date"]
display(deadline.dt.year.value_counts().sort_index().rename("n_lines").to_frame())

after_data = items[deadline.dt.year >= 2019]
print("lines with a deadline in 2019 or later:", len(after_data), "in", after_data["order_id"].nunique(), "orders")
display(after_data[["order_id", "shipping_limit_date", "price"]])

# Do the lines of one order have different deadlines?
spread = items.groupby("order_id")["shipping_limit_date"].agg(["min", "max"])
spread_days = (spread["max"] - spread["min"]).dt.total_seconds() / 86400
print("orders with more than one distinct deadline:", (spread_days > 0).sum())
print("... median / maximum gap between deadlines (days):", round(spread_days[spread_days > 0].median(), 1), "/", round(spread_days.max(), 1))

,n_lines
shipping_limit_date,
2016,370
2017,49765
2018,62511
2020,4


lines with a deadline in 2019 or later: 4 in 3 orders


,order_id,shipping_limit_date,price
8643,13bdf405f961a6deec817d817f5c6624,2020-02-05 03:30:51,69.99
68516,9c94a4ea2f7876660fa6f1b59b69c8e6,2020-02-03 20:23:22,75.99
85729,c2bb89b5c1dd978d507284be78a04cb2,2020-04-09 22:35:08,99.99
85730,c2bb89b5c1dd978d507284be78a04cb2,2020-04-09 22:35:08,99.99


orders with more than one distinct deadline: 336
... median / maximum gap between deadlines (days): 3.0 / 25.0


- Deadlines fall in 2016 (370 lines), 2017 (49,765) and 2018 (62,511), plus 4 lines in 2020.
- The 4 lines in 2020 (3 orders) are after the end of the data (2018). These are errors or manual entries.